# 01 · Preparação dos dados 

**Hipótese:** bairro mais violento → metro quadrado mais barato?

### Fontes

| Fonte | Conteúdo | Granularidade |
|---|---|---|
| `Crimes/SPDadosCriminais_{2022..2026}.xlsx` | Boletins de ocorrência da SSP-SP | 1 linha por ocorrência, **estado inteiro** |
| `Imoveis/GUIAS*.xlsx` | Guias de ITBI pagas | 1 linha por transação, **só a capital** |

### Merge das bases

As duas bases não têm chave geográfica em comum:
- ITBI: nome abreviado da rua; `Bairro` preenchido em 61% das linhas.
- Crimes: bairro em 99,7%; lat/lon em 81%.

**Estratégia:** usar crimes (mais completo) como dicionário de bairros para o ITBI, agrupando por rua normalizada.

- Problema: via longa cruza vários bairros. Ex.: `BRIGADEIRO LUIS ANTONIO` tem só 49,7% de pureza cidade-toda.
- Solução: dicionário em **duas granularidades** — cidade-toda (fallback) e por `(rua, distrito)`, quando o crime tem coordenada.
- O distrito vem da geometria do GeoSampa, não de inferência.

```
dicionario_ruas['BRIGADEIRO LUIS ANTONIO']:
    cidade toda                  → BELA VISTA
    (·, distrito Jard. Paulista) → JARDIM PAULISTA
```

- Bairro-com-bairro cobriria só 31,5% das transações.
- Via nome de rua: 92,7%.

### Saídas

`crimes_sp.parquet` · `itbi_sp.parquet` · `dicionario_ruas.parquet` · `setor_para_distrito.parquet` · `distritos_sp.geojson` · `setores_fiscais.geojson` — parquets já tipados e limpos.

### Visão geral do pipeline

Montado aqui, consumido no merge final (`02_eda.ipynb`):

```
 SPDadosCriminais (estado)        GUIAS de ITBI (capital)          GeoSampa (WFS)
      │ filtra capital                  │ SQL[:3] = setor_fiscal          │ distrito_municipal, setor_fiscal
      ▼                                 ▼                                 ▼
 crimes_sp.parquet                itbi_sp.parquet              distritos_sp / setores_fiscais
 bairro 99,7% · lat/lon 81%       bairro_itbi 61,1% · setor_fiscal 100%    │
      │                                 │                                 │ overlay por área (seção 2)
      │ sjoin c/ distritos_sp (seção 3) │                                 ▼
      ▼                                 │                     setor_para_distrito.parquet
 crimes + distrito (80,8%)              │                     setor_fiscal → distrito, 100% exato
      │                                 │◄────────────────────────────────┘
      │ agrega por rua_key               │  setor_fiscal → distrito, 100% exato
      │ e por (rua_key, distrito)        │
      │ (seção 5)                       │
      ▼                                 ▼
 dicionario_ruas.parquet           itbi.distrito (exato, 100%)
 cidade-toda + por distrito              │
      │                                 │
      └────────────────┬────────────────┘
                        ▼
          localiza() — 02_eda.ipynb, seção 3
          1º Bairro do próprio ITBI ............. 38,9%
          2º propagação por CEP .................. 92,6%
          3º dicionário de ruas, por distrito primeiro,
             cidade-toda como resíduo ............ 79,8% (63,6% já vêm da versão por distrito)
                        ▼
          itbi.bairro + itbi.lat/lon — combinado 97,0%
```

In [5]:
import gzip
import io
import re
import unicodedata
import urllib.request
from pathlib import Path

import geopandas as gpd
import pandas as pd
from openpyxl import load_workbook

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CRIMES = RAIZ / "data" / "SP" / "Crimes"
IMOVEIS = RAIZ / "data" / "SP" / "Imoveis"
CACHE = RAIZ / "data" / "SP" / "cache"
CACHE.mkdir(exist_ok=True)

sorted(p.name for p in CRIMES.glob("*.xlsx")) + sorted(p.name for p in IMOVEIS.glob("*.xlsx"))

['SPDadosCriminais_2022.xlsx',
 'SPDadosCriminais_2023.xlsx',
 'SPDadosCriminais_2024.xlsx',
 'SPDadosCriminais_2025.xlsx',
 'SPDadosCriminais_2026.xlsx',
 'GUIAS DE ITBI PAGAS (28012026) XLS.xlsx',
 'GUIAS DE ITBI PAGAS (30092026).xlsx',
 'GUIAS-DE-ITBI-PAGAS-2023.xlsx',
 'GUIAS-DE-ITBI-PAGAS-2024.xlsx',
 'GUIAS_DE_ITBI_PAGAS_12-2022.xlsx']

In [6]:
TIPOS_VIA = {
    "R", "RUA", "AV", "AVENIDA", "TV", "TRAVESSA", "PC", "PRACA", "AL", "ALAMEDA",
    "EST", "ESTR", "ESTRADA", "LG", "LARGO", "VD", "VIADUTO", "VIA", "ROD", "RODOVIA",
    "MARGINAL", "BC", "BECO", "PSG", "PASSAGEM", "PASSEIO", "PTE", "PONTE", "ACS",
    "ACESSO", "COR", "CORREDOR", "ESC", "ESCADARIA", "TR", "TREVO", "BALAO", "COMP",
    "COMPLEXO", "PRQ", "PARQUE", "PQ", "JD", "JARDIM", "CJ", "CONJUNTO", "SIT", "MKD",
}

ABREVIACOES = {
    "S": "SAO", "STO": "SANTO", "STA": "SANTA", "D": "DOM", "PE": "PADRE",
    "PROF": "PROFESSOR", "PROFA": "PROFESSORA", "DR": "DOUTOR", "DRA": "DOUTORA",
    "CEL": "CORONEL", "GEN": "GENERAL", "CAP": "CAPITAO", "ENG": "ENGENHEIRO",
    "SEN": "SENADOR", "DEP": "DEPUTADO", "PRES": "PRESIDENTE", "CDE": "CONDE",
    "VISC": "VISCONDE", "BRIG": "BRIGADEIRO", "MAL": "MARECHAL", "REV": "REVERENDO",
    "MIN": "MINISTRO", "DES": "DESEMBARGADOR", "SGT": "SARGENTO", "TEN": "TENENTE",
    "MAJ": "MAJOR", "ALM": "ALMIRANTE", "N": "NOSSA", "SRA": "SENHORA",
    "MONS": "MONSENHOR", "VER": "VEREADOR", "COM": "COMENDADOR", "BR": "BARAO",
    "MQS": "MARQUES", "IRM": "IRMA", "CONS": "CONSELHEIRO", "PRINC": "PRINCESA",
}

VAZIOS = {"", "NULL", "NAN", "NONE"}


def sem_acento(texto):
    normalizado = unicodedata.normalize("NFKD", str(texto))
    return normalizado.encode("ascii", "ignore").decode().upper().strip()


def limpa_texto(valor):
    texto = sem_acento(valor)
    return None if texto in VAZIOS else texto


def chave_rua(nome):
    """Retira o tipo de via (`R`, `AV`, `TV`…) e expande as abreviações de título (`BRIG` → `BRIGADEIRO`). 
    
    O que sobra é o nome próprio da rua, possibilitando o join."""
    texto = limpa_texto(nome)
    if texto is None or "VEDACAO" in texto:
        return None
    palavras = re.sub(r"[^A-Z0-9 ]", " ", texto).split()
    while palavras and palavras[0] in TIPOS_VIA:
        palavras = palavras[1:]
    return " ".join(ABREVIACOES.get(p, p) for p in palavras) or None


def em_unicos(serie, funcao):
    return serie.map({valor: funcao(valor) for valor in serie.dropna().unique()})


def cache(nome, construir):
    destino = CACHE / f"{nome}.parquet"
    if destino.exists():
        print(f"{nome}: lendo do cache")
        return pd.read_parquet(destino)
    tabela = construir()
    tabela.to_parquet(destino, index=False)
    print(f"{nome}: {len(tabela):,} linhas gravadas em {destino.name}")
    return tabela


chave_rua("AV BRIG LUIS ANTONIO"), chave_rua("AVENIDA BRIGADEIRO LUÍS ANTÔNIO")

('BRIGADEIRO LUIS ANTONIO', 'BRIGADEIRO LUIS ANTONIO')

## 1 · Crimes

Lido célula por célula (não `pd.read_excel` direto):

- **Estado inteiro.** Só ~40% das linhas são da capital — filtra na leitura, não depois.
- **Cabeçalho muda a cada ano.** `DESCR_PERIODO`/`DESC_PERIODO`, `CIRCUNCRIÇÃO`/`CIRCUNSCRIÇÃO`, `CIDADE` em 2022. Coluna localizada por prefixo; se nenhuma bater, `acha_coluna` falha com erro explícito.
- **Tipos misturados.** `LATITUDE` traz float, `0` e `'NULL'` na mesma coluna — limpa antes de salvar o parquet.

Geografia usada: *circunscrição* (onde o fato ocorreu). Coordenadas fora de SP (lat -24,0 a -23,3 · lon -46,9 a -46,3) viram nulas.

In [7]:
CAMPOS_CRIME = {
    "municipio": ("NOME_MUNICIPIO_CIRCUN",),
    "data_ocorrencia": ("DATA_OCORRENCIA_BO",),
    "periodo": ("DESC_PERIODO", "DESCR_PERIODO"),
    "local": ("DESCR_SUBTIPOLOCAL",),
    "bairro": ("BAIRRO",),
    "logradouro": ("LOGRADOURO",),
    "lat": ("LATITUDE",),
    "lon": ("LONGITUDE",),
    "natureza": ("NATUREZA_APURADA",),
    "delegacia": ("NOME_DELEGACIA_CIRCUN",),
}

JANELA_CIDADE = {"lat": (-24.0, -23.3), "lon": (-46.9, -46.3)}


def acha_coluna(cabecalho, prefixos, aba):
    posicao = next((i for i, c in enumerate(cabecalho) if c.startswith(prefixos)), None)
    if posicao is None:
        raise KeyError(f"{aba}: nenhuma coluna começa com {prefixos}")
    return posicao


def ler_aba_crime(caminho, aba):
    planilha = load_workbook(caminho, read_only=True)
    linhas = planilha[aba].iter_rows(values_only=True)
    cabecalho = [sem_acento(c) for c in next(linhas)]
    posicao = {
        novo: acha_coluna(cabecalho, prefixos, aba)
        for novo, prefixos in CAMPOS_CRIME.items()
    }
    coluna_municipio = posicao["municipio"]
    dados = [
        [linha[i] for i in posicao.values()]
        for linha in linhas
        if linha[coluna_municipio] == "S.PAULO"
    ]
    planilha.close()
    return pd.DataFrame(dados, columns=list(posicao)).drop(columns="municipio")


def limpa_crimes(tabela):
    for coluna in ["bairro", "natureza", "periodo", "local", "delegacia"]:
        tabela[coluna] = em_unicos(tabela[coluna], limpa_texto).astype("category")

    tabela["logradouro"] = em_unicos(tabela["logradouro"], limpa_texto)
    tabela["rua_key"] = em_unicos(tabela["logradouro"], chave_rua)
    tabela["data_ocorrencia"] = pd.to_datetime(tabela["data_ocorrencia"], errors="coerce")
    tabela["ano"] = tabela["data_ocorrencia"].dt.year
    tabela["mes"] = tabela["data_ocorrencia"].dt.to_period("M").astype(str)

    for coluna, (minimo, maximo) in JANELA_CIDADE.items():
        valores = pd.to_numeric(tabela[coluna], errors="coerce")
        tabela[coluna] = valores.where(valores.between(minimo, maximo))

    return tabela[tabela["ano"].between(2022, 2026)].reset_index(drop=True)


def monta_crimes():
    partes = []
    for caminho in sorted(CRIMES.glob("SPDadosCriminais_*.xlsx")):
        planilha = load_workbook(caminho, read_only=True)
        abas = [a for a in planilha.sheetnames if re.fullmatch(r"(JAN-JUN|JUL-DEZ)_\d{4}", a)]
        planilha.close()
        for aba in abas:
            parte = ler_aba_crime(caminho, aba)
            print(f"  {aba}: {len(parte):,} ocorrências na capital")
            partes.append(parte)
    return limpa_crimes(pd.concat(partes, ignore_index=True))


crimes = cache("crimes_sp", monta_crimes)
crimes.shape

crimes_sp: lendo do cache


(2214258, 12)

In [8]:
crimes.notna().mean().mul(100).round(1).sort_values().to_frame("% preenchido")

,% preenchido
periodo,32.3
lat,81.0
lon,81.0
rua_key,91.9
local,99.5
bairro,99.7
logradouro,100.0
data_ocorrencia,100.0
natureza,100.0
delegacia,100.0


## 2 · Geometria da cidade (GeoSampa)

Baixada aqui porque dois pontos à frente precisam dela:
- Dicionário de ruas (seção 5) — saber o distrito de cada crime.
- ITBI — `setor_fiscal` (3 primeiros dígitos do `SQL`) é chave exata, sem inferência.

Três camadas:

| camada | unidades | uso |
|---|---|---|
| `distrito_municipal` | 96 | mapas, checagem do notebook 05, dicionário de ruas |
| `setor_fiscal` | 309 | chave exata do ITBI |
| `setor → distrito` | — | derivada, por maior sobreposição de área |

- `setor → distrito`: cada setor fica com o distrito que mais se sobrepõe nele — determinístico, sem escolha manual.
- Checagem de sanidade: setor 001 → Sé, 002/003 → Brás, 006 → República.

In [9]:
WFS_GEOSAMPA = "https://wms.geosampa.prefeitura.sp.gov.br/geoserver/geoportal/wfs"
CRS_METRICO = 31983
TOLERANCIA_M = 20

CAMADAS = {
    "distritos_sp": ("distrito_municipal", ["cd_distrito_municipal", "nm_distrito_municipal"]),
    "setores_fiscais": ("setor_fiscal", ["cd_setor_fiscal"]),
}


def baixa_camada(camada, colunas):
    consulta = (
        f"{WFS_GEOSAMPA}?service=WFS&version=1.0.0&request=GetFeature"
        f"&typeName={camada}&outputFormat=application/json&srsName=EPSG:4326"
    )
    pedido = urllib.request.Request(
        consulta, headers={"Accept-Encoding": "identity", "User-Agent": "Mozilla/5.0"}
    )
    with urllib.request.urlopen(pedido, timeout=300) as resposta:
        conteudo = resposta.read()
    if conteudo[:2] == b"\x1f\x8b":
        conteudo = gzip.decompress(conteudo)
    geometrias = gpd.read_file(io.BytesIO(conteudo))[colunas + ["geometry"]]
    geometrias["geometry"] = (
        geometrias.to_crs(CRS_METRICO).simplify(TOLERANCIA_M).to_crs(4326)
    )
    return geometrias


def cache_geo(nome):
    destino = CACHE / f"{nome}.geojson"
    if destino.exists():
        print(f"{nome}: lendo de {destino.name}")
        return gpd.read_file(destino)
    camada, colunas = CAMADAS[nome]
    geometrias = baixa_camada(camada, colunas)
    geometrias.to_file(destino, driver="GeoJSON")
    print(f"{nome}: {len(geometrias)} feições baixadas do GeoSampa ({destino.stat().st_size / 1024:,.0f} KB)")
    return geometrias


distritos = cache_geo("distritos_sp")
setores = cache_geo("setores_fiscais")
setores["setor_fiscal"] = setores["cd_setor_fiscal"].astype(str).str.zfill(3)
setores = setores.dissolve(by="setor_fiscal").reset_index()[["setor_fiscal", "geometry"]]

print(f"\n{len(distritos)} distritos · {len(setores)} setores fiscais")

distritos_sp: lendo de distritos_sp.geojson
setores_fiscais: lendo de setores_fiscais.geojson

96 distritos · 309 setores fiscais


In [10]:
def monta_setor_para_distrito():
    cruzamento = gpd.overlay(setores, distritos, how="intersection", keep_geom_type=True)
    cruzamento["area_m2"] = cruzamento.to_crs(CRS_METRICO).area
    return (
        cruzamento.sort_values("area_m2")
        .groupby("setor_fiscal")
        .agg(distrito=("nm_distrito_municipal", "last"), area_m2=("area_m2", "last"))
        .reset_index()
    )


setor_para_distrito = cache("setor_para_distrito", monta_setor_para_distrito)

print(f"\ndistritos alcançados: {setor_para_distrito['distrito'].nunique()} de {len(distritos)}")

setor_para_distrito: lendo do cache

distritos alcançados: 94 de 96


## 3 · Crimes → distrito

- Ponto-em-polígono contra `distritos_sp`, só nos 81% de crimes com lat/lon.
- Ausência de coordenada não é aleatória (ver `05_checagem_distrito.ipynb`) — mas aqui o uso é local à rua, não contagem agregada, o que limita o viés.
- `distrito` fica só em memória: não entra em `crimes_sp.parquet`, pra não mudar o contrato desse cache.

In [11]:
geolocalizados = crimes.dropna(subset=["lat", "lon"])
pontos = gpd.GeoDataFrame(
    {"indice": geolocalizados.index},
    geometry=gpd.points_from_xy(geolocalizados["lon"], geolocalizados["lat"]),
    crs=4326,
)
dentro = gpd.sjoin(pontos, distritos[["nm_distrito_municipal", "geometry"]], how="left", predicate="within")
mapa_distrito = dentro.drop_duplicates("indice").set_index("indice")["nm_distrito_municipal"]
crimes["distrito"] = crimes.index.map(mapa_distrito)

print(f"crimes com distrito resolvido: {crimes['distrito'].notna().mean():.1%}")

crimes com distrito resolvido: 80.8%


## 4 · ITBI

- **Sem cabeçalho no topo** — a planilha foi ordenada pela coluna A, e `Nº do Cadastro (SQL)` (texto) ordena depois de qualquer número. Lida sem cabeçalho, com os 28 nomes declarados à mão; mantém só linhas com a 1ª coluna numérica (descarta rodapé vazio de brinde).
- `SQL` = `setor (3) + quadra (3) + lote (4) + dígito (1)`.
- `setor_fiscal` (3 primeiros dígitos): 100% preenchido. Geografia de reserva, e usado para desambiguar o dicionário de ruas por distrito (seção 5).

In [12]:
COLUNAS_ITBI = [
    "sql", "logradouro", "numero", "complemento", "bairro_itbi", "referencia", "cep",
    "natureza_transacao", "valor_transacao", "data_transacao", "vvr",
    "proporcao_transmitida", "vvr_proporcional", "base_calculo", "tipo_financiamento",
    "valor_financiado", "cartorio", "matricula", "situacao_sql", "area_terreno",
    "testada", "fracao_ideal", "area_construida", "uso_cod", "uso_desc", "padrao_cod",
    "padrao_desc", "acc",
]

NUMERICAS_ITBI = [
    "valor_transacao", "vvr", "vvr_proporcional", "base_calculo", "valor_financiado",
    "proporcao_transmitida", "area_terreno", "testada", "fracao_ideal",
    "area_construida", "acc", "uso_cod", "padrao_cod",
]


def limpa_itbi(tabela):
    tabela[NUMERICAS_ITBI] =            tabela[NUMERICAS_ITBI].apply(pd.to_numeric, errors="coerce")
    tabela["sql"] =                     tabela["sql"].astype("int64").astype(str).str.zfill(11)
    tabela["setor_fiscal"] =            tabela["sql"].str[:3]
    tabela["logradouro"] =              em_unicos(tabela["logradouro"], limpa_texto)
    tabela["rua_key"] =                 em_unicos(tabela["logradouro"], chave_rua)
    tabela["bairro_itbi"] =             em_unicos(tabela["bairro_itbi"], limpa_texto)
    tabela["data_transacao"] =          pd.to_datetime(tabela["data_transacao"], errors="coerce")
    tabela["ano"] =                     tabela["data_transacao"].dt.year
    tabela["mes"] =                     tabela["data_transacao"].dt.to_period("M").astype(str)

    for coluna in ["natureza_transacao", "uso_desc", "padrao_desc", "setor_fiscal", "situacao_sql"]:
        tabela[coluna] = em_unicos(tabela[coluna], limpa_texto).astype("category")

    descartaveis = ["complemento", "referencia", "cartorio", "tipo_financiamento", "matricula"]
    return tabela.drop(columns=descartaveis).reset_index(drop=True)


def monta_itbi():
    partes = []
    for caminho in sorted(IMOVEIS.glob("*.xlsx")):
        planilha = load_workbook(caminho, read_only=True)
        abas = [a for a in planilha.sheetnames if re.fullmatch(r"[A-Z]{3}-\d{4}", a)]
        for aba in abas:
            linhas = planilha[aba].iter_rows(values_only=True)
            dados = [
                (list(linha) + [None] * 28)[:28]
                for linha in linhas
                if isinstance(linha[0], (int, float))
            ]
            partes.append(pd.DataFrame(dados, columns=COLUNAS_ITBI))
        planilha.close()
        print(f"  {caminho.name}: {len(abas)} meses")
    return limpa_itbi(pd.concat(partes, ignore_index=True))


itbi = cache("itbi_sp", monta_itbi)
itbi.shape

itbi_sp: lendo do cache


(943536, 27)

In [13]:
print(f"ITBI com setor fiscal presente na malha: {itbi['setor_fiscal'].astype(str).isin(set(setores['setor_fiscal'])).mean():.2%}\n")
itbi.notna().mean().mul(100).round(1).sort_values().to_frame("% preenchido")

ITBI com setor fiscal presente na malha: 100.00%



,% preenchido
bairro_itbi,61.1
acc,82.6
uso_desc,99.7
sql,100.0
cep,100.0
natureza_transacao,100.0
logradouro,100.0
numero,100.0
vvr,100.0
proporcao_transmitida,100.0


## 5 · Dicionário de ruas

Por rua normalizada: bairro mais frequente + coordenada mediana (mediana resiste a uma coordenada isolada errada).

- **Problema:** via longa cruza vários bairros. `RAPOSO TAVARES` tem 26,6% de pureza, `PRESIDENTE WILSON` 44,8% — quase metade das ruas do dicionário tem ocorrências em mais de um bairro.
- **Correção:** duas camadas.
  - Cidade-toda (fallback — rua sem coordenada, ou distrito que não bate com nenhum bucket conhecido).
  - Por `(rua_key, distrito)` — usa o `distrito` da seção 3.

In [14]:
def agrega_ruas(base, chaves):
    pares = base.groupby([*chaves, "bairro"], observed=True).size().rename("n").reset_index()
    modal = (
        pares.sort_values("n")
        .groupby(chaves, observed=True)
        .agg(bairro=("bairro", "last"), n_ocorrencias=("n", "sum"))
    )
    coordenadas = base.groupby(chaves, observed=True)[["lat", "lon"]].median()
    return modal.join(coordenadas).reset_index()


def monta_dicionario():
    base = crimes.dropna(subset=["rua_key", "bairro"])
    cidade_toda = agrega_ruas(base, ["rua_key"]).assign(distrito=None)
    por_distrito = agrega_ruas(base.dropna(subset=["distrito"]), ["rua_key", "distrito"])
    return pd.concat([cidade_toda, por_distrito], ignore_index=True)


dicionario_ruas = cache("dicionario_ruas", monta_dicionario)
dicionario_ruas.sort_values("n_ocorrencias", ascending=False).head(10)

dicionario_ruas: lendo do cache


,rua_key,bairro,n_ocorrencias,lat,lon,distrito
37511,PAULISTA,BELA VISTA,21239,-23.562018,-46.655907,NaN
11989,CRUZEIRO DO SUL,SANTANA,15517,-23.516237,-46.625053,NaN
14025,DO ESTADO,BOM RETIRO,11606,-23.551932,-46.625802,NaN
5395,AUGUSTA,CONSOLACAO,11025,-23.553313,-46.655047,NaN
12347,DA LUZ,BOM RETIRO,10790,-23.534972,-46.635300,NaN
33091,MARIO DE ANDRADE,BARRA FUNDA,10562,-23.527406,-46.664291,NaN
43751,SENADOR TEOTONIO VILELA,CIDADE DUTRA,10468,-23.704761,-46.700849,NaN
12223,DA CONSOLACAO,CONSOLACAO,10222,-23.549329,-46.653231,NaN
53720,AUGUSTA,CONSOLACAO,9795,-23.552863,-46.654179,CONSOLACAO
60588,CRUZEIRO DO SUL,SANTANA,9486,-23.514622,-46.625053,SANTANA


## 6 · Cobertura do dicionário de ruas

Fração das transações do ITBI que herdam um bairro — comparada com a alternativa óbvia (casar `Bairro` do ITBI direto com o dos crimes).

In [16]:
cobertura = pd.DataFrame(
    {
        "estratégia": ["dicionário de ruas", "campo bairro do ITBI", "setor fiscal"],
        "cobertura": [
            itbi["rua_key"].isin(set(dicionario_ruas["rua_key"])).mean(),
            itbi["bairro_itbi"].isin(set(crimes["bairro"].dropna().unique())).mean(),
            itbi["setor_fiscal"].notna().mean(),
        ],
    }
)
cobertura.style.format({"cobertura": "{:.1%}"}).hide(axis="index")

estratégia,cobertura
dicionário de ruas,92.7%
campo bairro do ITBI,31.5%
setor fiscal,100.0%


**Checagem:** a desambiguação por distrito realmente aumenta a pureza nas ruas que motivaram a correção?

In [17]:
def pureza(tabela, chaves):
    pares = tabela.groupby([*chaves, "bairro"], observed=True).size().rename("n").reset_index()
    total = pares.groupby(chaves, observed=True)["n"].sum()
    maximo = pares.groupby(chaves, observed=True)["n"].max()
    return (maximo / total).rename("pureza")


base_pureza = crimes.dropna(subset=["rua_key", "bairro"])
pureza_cidade = pureza(base_pureza, ["rua_key"])
ruas_fracas = pureza_cidade[pureza_cidade < 0.70].index

pureza_distrito = pureza(base_pureza.dropna(subset=["distrito"]), ["rua_key", "distrito"])
pureza_media_por_distrito = (
    pureza_distrito[pureza_distrito.index.get_level_values("rua_key").isin(ruas_fracas)]
    .groupby(level="rua_key")
    .mean()
)

comparacao = pd.DataFrame({
    "pureza_cidade_toda": pureza_cidade.loc[ruas_fracas],
    "pureza_media_por_distrito": pureza_media_por_distrito,
})

print(f"{len(ruas_fracas)} ruas com pureza < 70% na camada cidade-toda")
print(f"pureza média nessas ruas: {comparacao['pureza_cidade_toda'].mean():.1%} cidade-toda "
      f"→ {comparacao['pureza_media_por_distrito'].mean():.1%} desambiguando por distrito")
comparacao.sort_values("pureza_cidade_toda").head(10)

6761 ruas com pureza < 70% na camada cidade-toda
pureza média nessas ruas: 57.5% cidade-toda → 66.5% desambiguando por distrito


,pureza_cidade_toda,pureza_media_por_distrito
rua_key,,
PARTICULAR,0.086420,0.920707
TIETE,0.099480,0.744658
DOIS,0.128571,0.813378
BEIJA FLOR,0.151515,0.760037
RODOANEL MARIO COVAS,0.180556,0.701197
SAO MIGUEL,0.181410,0.662776
BARAO 116,0.183333,0.773324
DA MINA,0.187500,0.882270
TUCANO,0.193548,0.805556


In [18]:
pd.DataFrame(
    [
        ("crimes_sp", len(crimes), crimes["mes"].min(), crimes["mes"].max()),
        ("itbi_sp", len(itbi), itbi["mes"].min(), itbi["mes"].max()),
        ("dicionario_ruas", len(dicionario_ruas), "—", "—"),
    ],
    columns=["tabela", "linhas", "início", "fim"],
)

,tabela,linhas,início,fim
0,crimes_sp,2214258,2022-01,2026-08
1,itbi_sp,943536,1994-06,2026-09
2,dicionario_ruas,98722,—,—


### Ressalvas

- Uma rua só entra no dicionário se teve alguma ocorrência — enviesa levemente a amostra pra ruas mais movimentadas. No nível de bairro o efeito é pequeno (todo bairro tem ocorrências).
- A camada por distrito usa só as ocorrências com coordenada (81%) — reduz o problema da via longa, não elimina.

➡️ Próximo: `02_eda.ipynb`.